##Celda 1 - Instalar dependencias y subir los 3 archivos zip del dataset exportado de Roboflow


Los nombres son:
*   globulos_rojos_3_6.v3i.yolov8.zip
*  globulos_rojos_3_6.v4i.yolov8.zip
*  globulos_rojos_3_6.v5i.yolov8.zip

Estos dataset parten de las mismas 169 imagenes iniciales y luego las modifica de diferentes formas. Para el primer dataset, dentro de Roboflow se puso la opcion de flip (horizontal y vertical). Para la segunda rotation 90°; y para la tercera rotation 45°

In [ ]:
from google.colab import files

print('Subí los 3 zips (v3, v4, v5) - podés seleccionar los 3 juntos en el cuadro de diálogo')
uploaded_zips = files.upload()
print('Zips subidos:', list(uploaded_zips.keys()))

Subí los 3 zips (v3, v4, v5) - podés seleccionar los 3 juntos en el cuadro de diálogo


Saving globulos_rojos_3_6.v5i.yolov8.zip to globulos_rojos_3_6.v5i.yolov8.zip
Saving globulos_rojos_3_6.v4i.yolov8.zip to globulos_rojos_3_6.v4i.yolov8.zip
Saving globulos_rojos_3_6.v3i.yolov8.zip to globulos_rojos_3_6.v3i.yolov8.zip
Zips subidos: ['globulos_rojos_3_6.v5i.yolov8.zip', 'globulos_rojos_3_6.v4i.yolov8.zip', 'globulos_rojos_3_6.v3i.yolov8.zip']


##Celda 2 - Descomprime los zip de arriba.

In [ ]:
import zipfile
import os

# Carpeta donde vamos a descomprimir cada zip por separado
carpeta_temp = '/content/temp_versiones'
os.makedirs(carpeta_temp, exist_ok=True)

carpetas_extraidas = []

for nombre_zip in uploaded_zips.keys():
    destino = os.path.join(carpeta_temp, nombre_zip.replace('.zip', ''))
    os.makedirs(destino, exist_ok=True)
    with zipfile.ZipFile(nombre_zip, 'r') as zip_ref:
        zip_ref.extractall(destino)
    carpetas_extraidas.append(destino)
    print(f'{nombre_zip} descomprimido en {destino}')

print('\nCarpetas extraídas:', carpetas_extraidas)

globulos_rojos_3_6.v5i.yolov8.zip descomprimido en /content/temp_versiones/globulos_rojos_3_6.v5i.yolov8
globulos_rojos_3_6.v4i.yolov8.zip descomprimido en /content/temp_versiones/globulos_rojos_3_6.v4i.yolov8
globulos_rojos_3_6.v3i.yolov8.zip descomprimido en /content/temp_versiones/globulos_rojos_3_6.v3i.yolov8

Carpetas extraídas: ['/content/temp_versiones/globulos_rojos_3_6.v5i.yolov8', '/content/temp_versiones/globulos_rojos_3_6.v4i.yolov8', '/content/temp_versiones/globulos_rojos_3_6.v3i.yolov8']


##Celda 3 - Une los datasets, quita las 169 imagenes iniciales que se pisan entre un dataset y el otro y crea "dataset_combinado"

In [ ]:
import shutil

# Carpeta final combinada
dataset_combinado = '/content/dataset_combinado'

# Creamos la estructura estándar de YOLO: train/valid/test, cada uno con images/ y labels/
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{dataset_combinado}/{split}/images', exist_ok=True)
    os.makedirs(f'{dataset_combinado}/{split}/labels', exist_ok=True)

contador_total = {'train': 0, 'valid': 0, 'test': 0}

for carpeta in carpetas_extraidas:
    for split in ['train', 'valid', 'test']:
        origen_images = f'{carpeta}/{split}/images'
        origen_labels = f'{carpeta}/{split}/labels'

        if not os.path.exists(origen_images):
            continue

        for archivo in os.listdir(origen_images):
            # Le agregamos el nombre de la carpeta de origen como prefijo
            # para evitar que se pisen archivos con el mismo nombre entre versiones
            prefijo = os.path.basename(carpeta)
            nuevo_nombre = f'{prefijo}_{archivo}'
            shutil.copy(
                os.path.join(origen_images, archivo),
                os.path.join(dataset_combinado, split, 'images', nuevo_nombre)
            )
            contador_total[split] += 1

        for archivo in os.listdir(origen_labels):
            prefijo = os.path.basename(carpeta)
            nuevo_nombre = f'{prefijo}_{archivo}'
            shutil.copy(
                os.path.join(origen_labels, archivo),
                os.path.join(dataset_combinado, split, 'labels', nuevo_nombre)
            )

print('Imágenes combinadas por split:')
for split, cantidad in contador_total.items():
    print(f'  {split}: {cantidad} imágenes')

Imágenes combinadas por split:
  train: 1337 imágenes
  valid: 144 imágenes
  test: 72 imágenes


## Celda 4 - Genera el `data.yaml` del dataset combinado

Este archivo le dice a YOLO dónde están las imágenes (train/valid/test) y qué clases tiene que reconocer (`GB`, `GR`, `PQT`). El orden de las clases es el mismo que en el `data.yaml` original de Roboflow - es importante no cambiarlo.

In [ ]:
yaml_combinado = f'''train: {dataset_combinado}/train/images
val: {dataset_combinado}/valid/images
test: {dataset_combinado}/test/images

nc: 3
names: ['GB', 'GR', 'PQT']
'''

ruta_yaml_combinado = f'{dataset_combinado}/data.yaml'
with open(ruta_yaml_combinado, 'w') as f:
    f.write(yaml_combinado)

print(f'data.yaml combinado guardado en: {ruta_yaml_combinado}')
print('\nContenido:')
print(yaml_combinado)

data.yaml combinado guardado en: /content/dataset_combinado/data.yaml

Contenido:
train: /content/dataset_combinado/train/images
val: /content/dataset_combinado/valid/images
test: /content/dataset_combinado/test/images

nc: 3
names: ['GB', 'GR', 'PQT']



## Celda 5 - Comprime el dataset combinado en un único zip

Este zip queda con la misma estructura que cualquier dataset exportado de Roboflow (`train/`, `valid/`, `test/`, `data.yaml`). Es el archivo que después se sube en el **Notebook 2 (Entrenamiento)** - el notebook de entrenamiento no necesita saber que este dataset viene de combinar 3 versiones, lo recibe como un dataset normal.

In [ ]:
import shutil as shutil_zip

nombre_zip_final = 'dataset_final'
shutil_zip.make_archive(nombre_zip_final, 'zip', dataset_combinado)

print(f'Dataset combinado comprimido en: {nombre_zip_final}.zip')

Dataset combinado comprimido en: dataset_final.zip


## Celda 6 - Descarga el zip final a la computadora

In [ ]:
from google.colab import files

files.download(f'{nombre_zip_final}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>